# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omaradelahmed/fly-rank-internship1/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
**Task type: binary classification.**

Lane 2 (Refresh / Content Opportunity Scoring) is fundamentally a yes/no question about an
observed outcome: *"is this page declining or not?"* That rules out clustering (no grouping
question here) and rules out training on raw ranking (there's no ground-truth ordering, only
a labeled outcome).

In practice, the model's output — a predicted **probability** of decline — doubles as a
priority score: editors sort pages by that probability to build their refresh queue. So the
*task type* is classification, and the *action* it feeds is a ranking/prioritization decision.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df["trend_direction"].value_counts())
print(f"\nRows: {len(df):,} | Clients: {df['client_id'].nunique()}")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Rows: 30,000 | Clients: 32


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
**Target: `is_declining_label`** = 1 if `trend_direction == "down"`, else 0.

Where it comes from: `trend_direction` is derived from `trend_pct`, which is itself an
**observed** measurement — the percent change in search impressions between the last 30 days
and the previous 30 days. So the label traces back to something that actually happened
(traffic falling), not to a manual judgment call about *why* a page is struggling.

**Honest caveat:** the observed `trend_pct` is converted into a binary label with a fixed
threshold (`< -20%` → "down"). That threshold is itself a defined rule layered on top of an
observed number — a page at -19% and a page at -21% get opposite labels despite near-identical
reality. Worth remembering when interpreting errors near that boundary.

**Never a feature:** `trend_direction` and `trend_pct` are the label's source and must never
be used as model inputs — that would be label leakage.

In [3]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df[["content_id", "trend_pct", "trend_direction", "is_declining_label"]].head(8))

print("\nClass balance:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

             content_id  trend_pct trend_direction  is_declining_label
0  content_304f48230142      -41.4            down                   1
1  content_a1fb4e703a9e      -57.7            down                   1
2  content_9aa793d4d895      -60.9            down                   1
3  content_331d6c4de07b      -13.8          stable                   0
4  content_d99b7a2d90ca      -34.7            down                   1
5  content_d4084a4bc775      -38.9            down                   1
6  content_9a34b442b552      -92.3            down                   1
7  content_a63219c6e95a        0.6          stable                   0

Class balance:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*
**Primary metric: ROC-AUC.** It checks how well the model separates decliners from
non-decliners across all thresholds, which is the right first check before picking any single
cutoff.

**Action-tied metric: Precision@K.** Editors can only refresh a limited number of pages per
week (say, the top 50–100 ranked by predicted decline probability). Precision@K answers the
question that actually matters for the workflow: *"of the top K pages the model flags, how
many are truly declining?"*

**Baseline to beat:** classes are close to balanced (~54% declining / ~46% not), so a
majority-class baseline gets ~54% accuracy and ROC-AUC = 0.5 (random ranking). Any model
claiming value has to clear both.


In [4]:
base_rate = df["is_declining_label"].mean()

print(f"Base rate (share declining): {base_rate:.1%}")
print(f"Naive majority-class accuracy baseline: {max(base_rate, 1 - base_rate):.1%}")
print("Naive ROC-AUC baseline (random ranking): 0.500")


Base rate (share declining): 54.2%
Naive majority-class accuracy baseline: 54.2%
Naive ROC-AUC baseline (random ranking): 0.500


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
unit_cols = [
    "content_id", "client_id", "content_type", "word_count",
    "impressions_90d", "ctr", "avg_position", "engagement_rate",
    "days_since_last_update", "trend_pct", "trend_direction",
]

df[unit_cols].head(5)


,content_id,client_id,content_type,word_count,impressions_90d,ctr,avg_position,engagement_rate,days_since_last_update,trend_pct,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,3803,0.76,10.6,5.88,20,-41.4,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,15320,0.05,20.3,0.00,25,-57.7,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,12581,0.09,36.5,0.00,20,-60.9,down
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,11751,0.49,6.2,1.28,22,-13.8,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,19140,0.13,44.0,0.00,14,-34.7,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A simple rule sounds tempting — e.g. "flag pages with `avg_position` worse than 20" or
"flag pages with `ctr` below 0.1%." But no single signal cleanly separates declining pages
from stable ones: the medians and spreads below overlap heavily between the two classes on
every strong candidate signal. A threshold on any one of them would misclassify large numbers
of pages in both directions.

Real decline is driven by many interacting, client-specific, time-varying signals
(seasonality, content type, freshness, ranking position, engagement) — exactly the kind of
tangled pattern ML is suited to combine, and a hand-written if-statement is not.


In [6]:
for col in ["avg_position", "ctr", "engagement_rate", "days_since_last_update"]:
    print(f"\n--- {col} by is_declining_label ---")
    print(df.groupby("is_declining_label")[col].describe()[["mean", "50%", "std"]])



--- avg_position by is_declining_label ---
                         mean    50%        std
is_declining_label                             
0                   16.823060  10.05  17.327709
1                   15.936305  11.30  13.159382

--- ctr by is_declining_label ---
                        mean   50%       std
is_declining_label                          
0                   0.731611  0.04  4.473514
1                   0.324138  0.08  1.689773

--- engagement_rate by is_declining_label ---
                        mean  50%       std
is_declining_label                         
0                   2.649728  0.0  8.785062
1                   2.437193  0.0  7.885554

--- days_since_last_update by is_declining_label ---
                         mean   50%        std
is_declining_label                            
0                   42.372543  20.0  40.857322
1                   49.245788  20.0  42.832998


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.